# Model 3: K-Nearest Neighbors (KNN) Classifier

- **KNN Assumption**:
  The model assumes that similar instances are near each other in the feature space.


### Our method:

We will use a **Bag-of-Words (BoW)** approach to vectorize the content, and then apply the K-Nearest Neighbors (KNN) algorithm for classification. The steps are as follows:

1. **Preprocessing**:
   - Remove any row with incomplete data
   - Remove stop words.
   - Remove Punctuation
   - Devide the dataset into train and test.
   
2. **Vectorization**:
   - Create a BoW representation for each document.
   - We are only doing this on the training data, so if any new word is encountered in the test data, it will be ignored.

3. **Training and Testing**:
   - Fit the KNN model using the training data.
   - Use Euclidean distance for calculating the distance between instances.
   - Run the model on the test data.

4. **Evaluation**:
   - We are using 4 evaluation metrics:
     - Accuracy
     - Precision
     - Recall
     - F1-score

5. **K-Fold Cross-Validation**:
   - Perform k-fold cross-validation to evaluate the model.
   - Calculate the average performance metrics across all folds.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
#Reading the data from the csv file
path = "final_dataset.csv"
df = pd.read_csv(path)
print(df.head())


   id                                              title  \
0   0  آریان خان نے والد کی سب سے بڑی خوبی بتا دی، ا...   
1   1  اداکارہ صحیفہ جبار نے تیزی سے اپنا وزن کیسے کم...   
2   2  کرائم پٹرول کے نوجوان اداکار کی اچانک موت، بھا...   
3   3  دیپیکا کی بیٹی کے ہمراہ پہلی جھلک سوشل میڈیا پ...   
4   4  سدھارتھ ملہوترا ماروتی سوزوکی کی مقبول گاڑی ڈز...   

                                                link  \
0  https://www.express.pk/story/2732283/arian-kha...   
1  https://www.express.pk/story/2732277/adakara-s...   
2  https://www.express.pk/story/2732263/crime-pet...   
3  https://www.express.pk/story/2732253/deepika-k...   
4  https://www.express.pk/story/2732223/sidharthm...   

                                             content     gold_label  
0  بالی ووڈ کے سپر اسٹار شاہ رُخ خان کے بڑے بیٹے ...  entertainment  
1  پاکستان شوبز کی خوبرو اداکارہ و ماڈل صحیفہ جبا...  entertainment  
2  بھارت کے ٹیلی ویژن اداکار نیتن چوہان کی گھر سے...  entertainment  
3  ممبئی: بالی وڈ کی م

In [3]:
import json
import re


def load_stopwords(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

urdu_stopwords = load_stopwords('./stopwords-ur.json')  

def remove_stopwords(text):
    words = text.split()  
    words = [word for word in words if word not in urdu_stopwords] 
    return ' '.join(words)

def remove_punctuation(text):
    punctuation = r'[^\w\s' + re.escape('،۔؟؛ٔ‘’"")') + r']'
    text = re.sub(punctuation, '', text)  
    return text

def clean_text(text):
    text = remove_stopwords(text) 
    text = remove_punctuation(text) 
    return text




In [4]:


df['content'] = df['content'].apply(clean_text)




In [5]:
#separting the gold labels 
X = df.drop(columns=['gold_label'])  
y = df['gold_label'] 
#Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print(y_train)

X_train shape: (895, 4)
X_test shape: (224, 4)
y_train shape: (895,)
y_test shape: (224,)
184            sports
711     entertainment
486            sports
260             world
398          business
            ...      
466            sports
121          business
1044            world
1095            world
860            sports
Name: gold_label, Length: 895, dtype: object


In [6]:
class BagOfWords:
    def __init__(self):
        self.vocabulary = []
        self.indices = {}

    def fit(self, corpus):
        unique= set()
        for sentence in corpus:
            for word in sentence.split():
                unique.add(word)
        self.vocabulary = list(unique)
        self.indices = {word: idx for idx, word in enumerate(self.vocabulary)}

    def vectorize(self, sentence):
        vector = np.zeros(len(self.vocabulary)) 
        for word in sentence.split():
            if word in self.indices: 
                vector[self.indices[word]] += 1
        return vector

In [7]:
# # code here
bOw = BagOfWords()
bOw.fit(X_train['content'])
dataTrainVectorized = np.array([bOw.vectorize(sentence) for sentence in X_train['content']])
dataTestVectorized = np.array([bOw.vectorize(sentence) for sentence in X_test['content']])


In [8]:
class KNN:
  def __init__(self, k):
    self.k = k
    self.train_x = None
    self.train_y = None

  def euclidean_distance(self, x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2, axis=1))

  def manhattan_distance(self, x1, x2):
    return np.sum(np.abs(x1 - x2), axis=1)

  def fit(self, train_x, train_y):
    self.train_x = train_x
    self.train_y = train_y

  def get_neighbors(self, new_point, distancefunc):
    if distancefunc == self.euclidean_distance:
        distances = self.euclidean_distance(new_point, self.train_x)
    elif distancefunc == self.manhattan_distance:
        distances = self.manhattan_distance(new_point, self.train_x)

    nearest_indices = np.argsort(distances)[:self.k]
    neighbors = self.train_y.iloc[nearest_indices].values
    return neighbors

  def predict(self, test_x, distancefunc):
    predictions = []
    for point in test_x:
        neighbors = self.get_neighbors(point, distancefunc)
        class_counts = {}
        for neighbor in neighbors:
            if neighbor in class_counts:
                class_counts[neighbor] += 1
            else:
                class_counts[neighbor] = 1
        most_common = max(class_counts, key=class_counts.get)
        predictions.append(most_common)
    return predictions

In [9]:
knn_model = KNN(k=4)
knn_model.fit(dataTrainVectorized, y_train)

y_pred_knn = knn_model.predict(dataTestVectorized, knn_model.euclidean_distance)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn, average='weighted')
recall_knn = recall_score(y_test, y_pred_knn, average='weighted')
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')

print(f"KNN Accuracy: {accuracy_knn}")
print(f"KNN Precision: {precision_knn}")
print(f"KNN Recall: {recall_knn}")
print(f"KNN F1-score: {f1_knn}")

KNN Accuracy: 0.7857142857142857
KNN Precision: 0.8064336664548754
KNN Recall: 0.7857142857142857
KNN F1-score: 0.7888676092043182


In [10]:
def k_fold_cross_validation(X, y, k, model, distancefunc):
    fold_size = len(X) // k
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    
    accuracy_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    
    for i in range(k):
        test_indices = indices[i * fold_size:(i + 1) * fold_size]
        train_indices = np.concatenate((indices[:i * fold_size], indices[(i + 1) * fold_size:]))
        
        X_train, X_test = X[train_indices], X[test_indices]
        y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test, distancefunc)
        
        accuracy_scores.append(accuracy_score(y_test, y_pred))
        precision_scores.append(precision_score(y_test, y_pred, average='weighted'))
        recall_scores.append(recall_score(y_test, y_pred, average='weighted'))
        f1_scores.append(f1_score(y_test, y_pred, average='weighted'))
    
    avg_accuracy = np.mean(accuracy_scores)
    avg_precision = np.mean(precision_scores)
    avg_recall = np.mean(recall_scores)
    avg_f1 = np.mean(f1_scores)
    
    return avg_accuracy, avg_precision, avg_recall, avg_f1

In [ ]:
k = 4
knn_model = KNN(k=4)
avg_accuracy, avg_precision, avg_recall, avg_f1 = k_fold_cross_validation(dataTrainVectorized, y_train, k, knn_model, knn_model.euclidean_distance)

print(f"K-Fold Cross-Validation Accuracy: {avg_accuracy}")
print(f"K-Fold Cross-Validation Precision: {avg_precision}")
print(f"K-Fold Cross-Validation Recall: {avg_recall}")
print(f"K-Fold Cross-Validation F1-score: {avg_f1}")

K-Fold Cross-Validation Accuracy: 0.7651685393258427
K-Fold Cross-Validation Precision: 0.8077067221092378
K-Fold Cross-Validation Recall: 0.7651685393258427
K-Fold Cross-Validation F1-score: 0.7705550775004049
